# Active Learning PLM Distillation — Colab Demo

Walks through the full pipeline on **DISPEF-M (subset)** without AWS or ESM3:

1. Install dependencies & clone repo  
2. Download & preprocess DISPEF-M  
3. **Explore data structure** — NPZ residue arrays, PyG graph object, Cα backbone vis  
4. **Explore model architecture** — Schake v2 dual-range GNN, parameter count, forward pass  
5. **Loss functions** — soft CE, KL-div, DSSP hard CE  
6. Generate mock teacher labels (DSSP-based, no ESM3 needed)  
7. Train the student GNN  
8. Evaluate & visualise results  

> **Runtime**: Runtime → Change runtime type → **T4 GPU** (free tier is fine)

## 0 · GPU check

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('CUDA ver:', torch.version.cuda)

## 1 · Install dependencies

`torch-scatter` must match the installed PyTorch + CUDA build — the cell detects both automatically.

In [ ]:
import torch, sys

torch_ver = torch.__version__.split('+')[0]      # e.g. '2.4.0'
cuda_ver  = torch.version.cuda.replace('.', '')  # e.g. '121'
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html'
print('PyG wheel index:', pyg_url)

!pip install -q torch-geometric
!pip install -q torch-scatter torch-cluster -f {pyg_url}
!pip install -q biopython 'mdtraj==1.10.1' pyyaml tqdm scikit-learn matplotlib

## 2 · Clone repository

In [ ]:
import os, sys

REPO_URL = 'https://github.com/hxyue1/active-learning-plm-distillation.git'
REPO_DIR = '/content/active-learning-plm-distillation'
BRANCH   = 'colab-dev'

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())

## 3 · Workspace paths, subset size, and runtime config

Change `SUBSET` to control how many proteins to preprocess:  
- `200` → fast sanity check (~2 min preprocess, ~1 min/epoch on T4)  
- `0` → full DISPEF-M (~7 000 proteins, matches paper exactly)

In [ ]:
import os, yaml, pathlib

WORKSPACE     = '/content/dispef_ws'
RAW_ROOT      = f'{WORKSPACE}/data/raw/dispef'
PROCESSED     = f'{WORKSPACE}/data/processed'
TEACHER_CACHE = f'{WORKSPACE}/cache/teacher'
CHECKPOINTS   = f'{WORKSPACE}/checkpoints'
OUTPUTS       = f'{WORKSPACE}/outputs'
LOGS          = f'{WORKSPACE}/logs'
DATASET_NAME  = 'dispef_m'

SUBSET   = 200   # set to 0 for full DISPEF-M
RUN_NAME = 'colab_demo'

for d in [RAW_ROOT, PROCESSED, TEACHER_CACHE, CHECKPOINTS, OUTPUTS, LOGS]:
    os.makedirs(d, exist_ok=True)

# Write a runtime config that injects the correct workspace paths.
# This avoids editing the repo config and works even if WORKSPACE changes.
base_cfg = yaml.safe_load(open(f'{REPO_DIR}/configs/colab_demo.yaml'))
base_cfg['paths'].update({
    'project_root'       : WORKSPACE,
    'raw_dispef_root'    : RAW_ROOT,
    'processed_root'     : PROCESSED,
    'teacher_cache_root' : TEACHER_CACHE,
    'outputs_root'       : OUTPUTS,
    'checkpoints_root'   : CHECKPOINTS,
    'logs_root'          : LOGS,
})
RUNTIME_CFG = '/tmp/colab_runtime.yaml'
with open(RUNTIME_CFG, 'w') as f:
    yaml.dump(base_cfg, f)

print('Workspace   :', WORKSPACE)
print('Runtime cfg :', RUNTIME_CFG)

## 4 · Download DISPEF-M from Zenodo

Zenodo record **13755810** is specifically the DISPEF-**M** (medium) subset.  
This is **not** the full DISPEF dataset — only the M split used in the paper.

In [ ]:
import glob

ARCHIVE = f'{RAW_ROOT}/zenodo_13755810.zip'

if not os.path.exists(ARCHIVE):
    print('Downloading DISPEF-M from Zenodo...')
    !curl -L 'https://zenodo.org/api/records/13755810/files-archive' -o {ARCHIVE}
else:
    print('Archive already present')

if not glob.glob(f'{RAW_ROOT}/*DISPEF_M_tr.pt'):
    print('Extracting...')
    !unzip -q {ARCHIVE} -d {RAW_ROOT}

print('\nFiles in raw root:')
!ls -lh {RAW_ROOT}

## 5 · Preprocess DISPEF-M → per-protein NPZ files

**What this step does:**

```
DISPEF_M_tr.pt / DISPEF_M_te.pt  (raw atom tensors)
        ↓
  extract backbone N, CA, C per residue  →  coords [L, 3, 3]  (nm)
  infer 1-letter AA from atom names      →  aa_idx [L]
  compute DSSP 8-class via mdtraj        →  dssp_idx [L]
        ↓
  one <sample_id>.npz per protein
  splits.json  (train / val / test)
  manifest.csv
```

`--max-files N` caps total proteins processed (train + test combined).

In [ ]:
import json

max_flag = f'--max-files {SUBSET}' if SUBSET > 0 else ''

!python -m data.preprocess_dispef \
    --raw-root       {RAW_ROOT}  \
    --processed-root {PROCESSED} \
    --dataset-name   {DATASET_NAME} \
    --val-fraction 0.1 --seed 42 \
    {max_flag}

splits = json.loads(open(f'{PROCESSED}/{DATASET_NAME}/splits.json').read())
print('\nSplit sizes:')
for k, v in splits.items():
    if isinstance(v, list):
        print(f'  {k:20s}: {len(v)}')

## 6 · Explore data structure

### 6a · Raw NPZ content (residue-level)

In [ ]:
import numpy as np, pathlib

proteins_dir = pathlib.Path(f'{PROCESSED}/{DATASET_NAME}/proteins')
sample_path  = sorted(proteins_dir.glob('*.npz'))[0]
sample       = dict(np.load(sample_path, allow_pickle=True))

print(f'Sample ID : {sample["sample_id"]}')
seq_str = ''.join(str(c) for c in sample['sequence'].tolist())
print(f'Sequence  : {seq_str[:60]}...  (len={len(seq_str)})')
print()
print('Array            Shape                 dtype    Notes')
print('-'*70)
print(f'aa_idx           {str(sample["aa_idx"].shape):<22}{str(sample["aa_idx"].dtype):<9}'
      f'residue index 0-20  (20 AAs + X=unknown)')
print(f'coords           {str(sample["coords"].shape):<22}{str(sample["coords"].dtype):<9}'
      f'[L, 3 atoms, 3 xyz] in nm')
print(f'dssp_idx         {str(sample["dssp_idx"].shape):<22}{str(sample["dssp_idx"].dtype):<9}'
      f'SS8 class 0-7: G H I T E B S C  (-100=missing)')
print(f'atom_order       {str(sample["atom_order"].shape):<22}str      '
      f'{list(sample["atom_order"])}  (axis-1 order of coords)')

# SS8 distribution for this protein
from data.constants import INDEX_TO_SS8, SS8_CLASSES
d = sample['dssp_idx']
valid = d[d >= 0]
u, c = np.unique(valid, return_counts=True)
print('\nDSSP class distribution (this protein):')
for cls, cnt in zip(u, c):
    bar = '#' * int(cnt / max(c) * 30)
    print(f'  {INDEX_TO_SS8[cls]} ({cls}): {cnt:4d}  {bar}')

### 6b · PyG graph Data object (node-level)

The graph builder **expands** residue-level arrays to per-node arrays by repeating each residue 3× — one entry per backbone atom N, CA, C.  
A protein with **L** residues → a graph with **3L nodes**.

```
Residue i  →  node 3i   (N atom)
           →  node 3i+1 (CA atom)
           →  node 3i+2 (C atom)
```

The `node_to_residue` tensor makes it easy to aggregate back to residue level.

In [ ]:
from data.graph_builder import build_graph_data
import torch

graph = build_graph_data(
    sample_path=sample_path,
    teacher_path=None,
    cutoff=0.001,    # near-zero → no dataset-level edges (Schake builds its own)
    max_neighbors=0,
    center_coords=True,
)

L = graph.num_residues
print(f'Protein        : {graph.sample_id}')
print(f'Residues L     : {L}')
print(f'Nodes 3L       : {graph.pos.shape[0]}  (one per backbone atom N/CA/C)')
print()
print('Tensor              Shape          Description')
print('-'*65)
print(f'pos                 {str(graph.pos.shape):<15}x,y,z in nm — zero-centred')
print(f'aa_idx              {str(graph.aa_idx.shape):<15}residue AA index, repeated 3×')
print(f'atom_idx            {str(graph.atom_idx.shape):<15}atom type: 0=N  1=CA  2=C, tiled')
print(f'dssp_idx            {str(graph.dssp_idx.shape):<15}SS8 label, repeated 3×')
print(f'node_to_residue     {str(graph.node_to_residue.shape):<15}node index → residue index')
print(f'edge_index          {str(graph.edge_index.shape):<15}empty (Schake builds internally)')
print(f'edge_attr           {str(graph.edge_attr.shape):<15}empty')
print()
print('atom_idx pattern for first 3 residues (9 nodes):')
print(' ', graph.atom_idx[:9].tolist(), '  → [N, CA, C, N, CA, C, N, CA, C, ...]')
print('node_to_residue for first 9 nodes:')
print(' ', graph.node_to_residue[:9].tolist(), '  → [0, 0, 0, 1, 1, 1, 2, 2, 2, ...]')

### 6c · Backbone 3-D visualisation (Cα trace, coloured by SS8)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

SS8_COLORS = ['#e41a1c','#377eb8','#4daf4a','#984ea3',
               '#ff7f00','#a65628','#f781bf','#aaaaaa']
SS8_FULL   = ['G 3-10 helix','H α-helix','I π-helix','T turn',
               'E β-strand','B β-bridge','S bend','C coil']

ca_mask  = graph.atom_idx == 1            # CA atoms only
ca_pos   = graph.pos[ca_mask].numpy()     # [L, 3]
dssp_ca  = graph.dssp_idx[ca_mask].numpy()

fig = plt.figure(figsize=(13, 4))

# --- 3-D backbone ---
ax = fig.add_subplot(131, projection='3d')
ax.plot(ca_pos[:,0], ca_pos[:,1], ca_pos[:,2], '-', lw=0.4, color='#ddd', zorder=0)
for cls in range(8):
    m = dssp_ca == cls
    if m.any():
        ax.scatter(ca_pos[m,0], ca_pos[m,1], ca_pos[m,2],
                   s=12, c=SS8_COLORS[cls], label=SS8_FULL[cls], zorder=2)
ax.set_xlabel('x (nm)', fontsize=8); ax.set_ylabel('y (nm)', fontsize=8)
ax.set_zlabel('z (nm)', fontsize=8); ax.tick_params(labelsize=6)
ax.set_title(f'{graph.sample_id}\nCα trace', fontsize=9)
ax.legend(fontsize=5, loc='upper left', bbox_to_anchor=(1.0, 1.0))

# --- SS8 bar chart ---
ax2 = fig.add_subplot(132)
valid = dssp_ca[dssp_ca >= 0]
counts = [np.sum(valid == i) for i in range(8)]
ax2.bar(SS8_CLASSES, counts, color=SS8_COLORS)
ax2.set_xlabel('SS8'); ax2.set_ylabel('Residues')
ax2.set_title('DSSP distribution', fontsize=9)

# --- Pairwise Cα distance matrix (first 60 residues) ---
ax3 = fig.add_subplot(133)
n = min(60, len(ca_pos))
dmat = np.linalg.norm(ca_pos[:n, None, :] - ca_pos[None, :n, :], axis=-1)
im = ax3.imshow(dmat, cmap='viridis_r', vmin=0, vmax=2.5)
plt.colorbar(im, ax=ax3, label='nm')
ax3.set_title(f'Cα distance matrix\n(first {n} res)', fontsize=9)
ax3.set_xlabel('Residue'); ax3.set_ylabel('Residue')

plt.tight_layout()
plt.show()

## 7 · Model architecture — Schake v2

### 7a · How Schake v2 works

```
Input: pos [3L, 3]  aa_idx [3L]  atom_idx [3L]
          │
  Index remapping:
    atom_idx [N=0, CA=1, C=2] → Schake vocab [N=63, CA=1, C=0]
    aa_idx   [A=0, R=1, ...]  → Schake vocab [A=0, C=1, D=2, ...] (alphabetical)
          │
  Build two radius graphs on-the-fly each forward pass:
    SAKE branch  :  all atoms,  0.25 – 1.0 nm  (local / covalent geometry)
    SchNet branch:  CA only,    1.0  – 2.5 nm  (inter-residue contacts)
          │
  2 rounds of parallel message passing:
    h ← LayerNorm( h + SAKE_msg(h) + SchNet_msg(h) )
          │
  3-layer MLP output head → 8 SS8 logits per node
```

**Why two branches?**  
SAKE short-range captures bond angles and local backbone conformation; SchNet long-range captures tertiary contacts (β-sheet pairings, α-helix packing) that disambiguate SS8 class.

In [ ]:
import yaml
from models.factory import build_model

cfg   = yaml.safe_load(open(RUNTIME_CFG))
model = build_model(cfg)

# Parameter breakdown
n_total   = sum(p.numel() for p in model.parameters())
n_train   = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_fixed   = n_total - n_train
print(f'Total parameters    : {n_total:>8,}')
print(f'Trainable           : {n_train:>8,}')
print(f'Fixed (frozen)      : {n_fixed:>8,}')
print()

# Named modules for inspection
print('Named submodules:')
for name, mod in model.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f'  {name:<30} {n:>8,} params  {type(mod).__name__}')

### 7b · Single forward pass (sanity check)

In [ ]:
import torch

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = model.to(device)
graph_dev = graph.to(device)

model.eval()
with torch.no_grad():
    logits = model(graph_dev)   # [3L, 8]

probs = torch.softmax(logits, dim=-1)
pred  = logits.argmax(dim=-1)

print(f'Input  pos shape : {graph_dev.pos.shape}  →  {graph_dev.num_residues} residues × 3 atoms')
print(f'Output logits    : {logits.shape}  →  {graph_dev.num_residues} residues × 3 atoms × 8 SS8 classes')
print()
print('SS8 classes : G  H  I  T  E  B  S  C')
print('Pred (N/CA/C for first 3 residues):')
for node_i in range(min(9, logits.shape[0])):
    atom = ['N', 'CA', 'C'][node_i % 3]
    res_i = node_i // 3
    true_ss = INDEX_TO_SS8.get(int(graph_dev.dssp_idx[node_i].item()), '?')
    pred_ss = ['G','H','I','T','E','B','S','C'][int(pred[node_i].item())]
    conf    = probs[node_i, pred[node_i]].item()
    print(f'  res {res_i} {atom:2s}: pred={pred_ss} (conf={conf:.2f})  true={true_ss}')

### 7c · Distillation loss functions

Training minimises a **combined loss**:

$$\mathcal{L} = \lambda_{\text{teacher}} \cdot \underbrace{-\sum_c p^{\text{teacher}}_c \log p^{\text{student}}_c}_{\text{soft CE}} + \lambda_{\text{DSSP}} \cdot \underbrace{-\log p^{\text{student}}_{y_{\text{DSSP}}}}_{\text{hard CE}}$$

- **Soft CE** (`lambda_teacher=1.0`): match the full ESM3 SS8 probability distribution — preserves uncertainty at helix/coil boundaries  
- **DSSP hard CE** (`lambda_dssp=1.0`): auxiliary supervision from mdtraj DSSP labels — grounds predictions in physics  
- **KL divergence**: alternative to soft CE; equivalent up to a constant when temperature=1

In [ ]:
from train.losses import soft_target_cross_entropy, teacher_kl_divergence, hard_cross_entropy
import torch

# Random mock teacher (for illustration only; real = ESM3 output)
mock_teacher = torch.softmax(torch.randn_like(logits), dim=-1)
dssp_targets = graph_dev.dssp_idx   # [3L], ignore_index=-100

l_soft_ce = soft_target_cross_entropy(logits, mock_teacher)
l_kl      = teacher_kl_divergence(logits, mock_teacher, temperature=1.0)
l_dssp    = hard_cross_entropy(logits, dssp_targets, ignore_index=-100)

lam_t, lam_d = 1.0, 1.0
total = lam_t * l_soft_ce + lam_d * l_dssp

print('Loss (random mock teacher — for illustration):')
print(f'  soft_ce (teacher)  = {l_soft_ce.item():.4f}   ← matches ESM3 distribution')
print(f'  kl_div  (teacher)  = {l_kl.item():.4f}   ← alternative; equal at T=1')
print(f'  hard_ce (DSSP)     = {l_dssp.item():.4f}   ← auxiliary signal')
print(f'  total              = {total.item():.4f}   = {lam_t}×soft_ce + {lam_d}×dssp_ce')

## 8 · Generate mock teacher labels

Real distillation requires ESM3 SS8 predictions cached by `teacher/generate_teacher_labels.py` (needs a large GPU or Forge API key).  
For Colab we generate **mock teacher labels** from DSSP instead — one-hot vectors (with optional label smoothing) in the same `.npz` format the trainer expects.

With mock labels the soft-CE loss reduces to standard hard-label CE, so training becomes supervised DSSP classification.  This is sufficient to verify the full pipeline and understand the code.

In [ ]:
import numpy as np, glob

!python scripts/generate_mock_teacher.py \
    --processed-root     {PROCESSED}     \
    --dataset-name       {DATASET_NAME}  \
    --teacher-cache-root {TEACHER_CACHE} \
    --label-smoothing 0.05

cache_files = sorted(glob.glob(f'{TEACHER_CACHE}/{DATASET_NAME}/*.npz'))
print(f'\nCache files written : {len(cache_files)}')

# Inspect one cached file
ex = np.load(cache_files[0])
tp = ex['teacher_probs_node']   # [3L, 8]
print(f'teacher_probs_node  : shape={tp.shape}  dtype={tp.dtype}')
print(f'Row sum check       : {tp[0].sum():.5f}  (should be 1.0)')
print(f'Example (node 0)    : {tp[0].round(3)}')
print('  → one-hot (with 0.05 smoothing) → near 1.0 on DSSP class, ~0.007 elsewhere')

## 9 · Train the student GNN

Config: `colab_demo.yaml` — 10 epochs, batch 32, no S3/W&B.  
Set `epochs: 120` in the config to reproduce the paper result (requires real ESM3 teacher labels).

In [ ]:
!python -m train.train --config {RUNTIME_CFG} --run-name {RUN_NAME}

## 10 · Evaluate on test set

In [ ]:
import pathlib, glob as _glob, json

# Run dir has a timestamp suffix: find the most recent one for this run name
run_dirs = sorted(_glob.glob(f'{OUTPUTS}/{RUN_NAME}_*'))
if not run_dirs:
    raise FileNotFoundError(f'No run dir found under {OUTPUTS}. Did training complete?')
run_dir  = pathlib.Path(run_dirs[-1])
ckpt_dir = pathlib.Path(CHECKPOINTS) / run_dir.name

best_ckpt = ckpt_dir / 'best.pt'
if not best_ckpt.exists():
    best_ckpt = ckpt_dir / 'last.pt'

print('Run dir  :', run_dir)
print('Ckpt     :', best_ckpt)

EVAL_OUT = f'{OUTPUTS}/eval/{run_dir.name}'

!python -m eval.evaluate \
    --config     {RUNTIME_CFG} \
    --checkpoint {best_ckpt}   \
    --split test               \
    --output-dir {EVAL_OUT}

In [ ]:
# eval.evaluate writes eval_summary_<split>.json
summary_path = pathlib.Path(EVAL_OUT) / 'eval_summary_test.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('Test set evaluation:')
    for k, v in summary.items():
        if isinstance(v, float):
            print(f'  {k:<30s}: {v:.4f}')
        else:
            print(f'  {k:<30s}: {v}')

## 11 · Training curves

In [ ]:
import csv, matplotlib.pyplot as plt

history_path = run_dir / 'history.csv'
if not history_path.exists():
    print('history.csv not found — training may not have completed')
else:
    with open(history_path) as f:
        rows = list(csv.DictReader(f))

    epochs    = [int(r['epoch'])                    for r in rows]
    train_ce  = [float(r['train_teacher_ce'])        for r in rows]
    val_ce    = [float(r['val_teacher_ce'])          for r in rows]
    train_acc = [float(r['train_teacher_top1_acc'])  for r in rows]
    val_acc   = [float(r['val_teacher_top1_acc'])    for r in rows]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, train_ce,  label='train', marker='o', ms=3)
    axes[0].plot(epochs, val_ce,    label='val',   marker='s', ms=3)
    axes[0].set_title('Soft Cross-Entropy (teacher)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, train_acc, label='train', marker='o', ms=3)
    axes[1].plot(epochs, val_acc,   label='val',   marker='s', ms=3)
    axes[1].set_title('Teacher Top-1 Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'{run_dir.name}', fontsize=11)
    plt.tight_layout()
    plt.show()

## 12 · Per-class accuracy (test set)

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, yaml, pathlib
from data.pyg_dataset import DistillationGraphDataset
from torch_geometric.loader import DataLoader as PyGLoader
from models.factory import build_model
from data.constants import SS8_CLASSES

cfg    = yaml.safe_load(open(RUNTIME_CFG))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

test_ds = DistillationGraphDataset(
    processed_root=pathlib.Path(PROCESSED),
    dataset_name=DATASET_NAME,
    split_name='test',
    teacher_root=pathlib.Path(TEACHER_CACHE) / DATASET_NAME,
    cutoff=float(cfg['graph']['cutoff']),
    max_neighbors=int(cfg['graph']['max_neighbors']),
)
loader = PyGLoader(test_ds, batch_size=16, shuffle=False)

model = build_model(cfg).to(device)
ckpt  = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

correct = np.zeros(8, dtype=np.float64)
total   = np.zeros(8, dtype=np.float64)

with torch.no_grad():
    for batch in loader:
        batch   = batch.to(device)
        logits  = model(batch)
        pred    = logits.argmax(dim=-1).cpu().numpy()
        true    = batch.dssp_idx.cpu().numpy()
        valid   = true >= 0
        for c in range(8):
            mask      = valid & (true == c)
            total[c]  += mask.sum()
            correct[c]+= (pred[mask] == c).sum()

per_class = np.where(total > 0, correct / total, np.nan)
macro_acc = np.nanmean(per_class)

SS8_COLORS = ['#e41a1c','#377eb8','#4daf4a','#984ea3',
               '#ff7f00','#a65628','#f781bf','#aaaaaa']

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(SS8_CLASSES, np.nan_to_num(per_class), color=SS8_COLORS)
ax.axhline(macro_acc, color='k', ls='--', lw=1, label=f'macro avg={macro_acc:.3f}')
ax.set_ylim(0, 1.05)
ax.set_xlabel('SS8 class'); ax.set_ylabel('Per-class accuracy (DSSP)')
ax.set_title('Test-set per-class accuracy (vs DSSP hard labels)')
ax.legend()
for i, (cls, acc) in enumerate(zip(SS8_CLASSES, per_class)):
    if not np.isnan(acc):
        ax.text(i, acc + 0.02, f'{acc:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Per-class accuracy:')
for cls, acc, n in zip(SS8_CLASSES, per_class, total.astype(int)):
    print(f'  {cls}  {acc:.3f}  (n={n})')
print(f'  macro avg: {macro_acc:.3f}')

---
## Summary

| | Detail |
|---|---|
| **Data structure** | Each protein: `npz` with residue-level `aa_idx [L]`, `coords [L,3,3]` (nm), `dssp_idx [L]`. Graph builder expands to 3L nodes (N/CA/C), attaching `pos`, `aa_idx`, `atom_idx`, `dssp_idx`, `node_to_residue`. |
| **Model** | Schake v2 — dual-range GNN. SAKE branch (0.25–1.0 nm, all atoms) captures local backbone geometry; SchNet branch (1.0–2.5 nm, CA-only) captures tertiary contacts. 2 message-passing rounds, 3-layer MLP head → 8 SS8 logits. |
| **Distillation** | `λ_teacher × soft_CE(student, ESM3)  +  λ_DSSP × hard_CE(student, DSSP)`. Soft CE preserves teacher uncertainty; DSSP CE provides hard physical grounding. |
| **Mock teacher** | `scripts/generate_mock_teacher.py` converts DSSP hard labels to smoothed one-hot teacher probs — lets you run the full pipeline without ESM3. |
| **Active learning** | `SplitIndex` in `data/pyg_dataset.py` manages `train / val / test / pool_unassigned` in `splits.json` for future AL cycles. |

**To run the real paper baseline (on AWS):**
```bash
bash scripts/generate_teacher_labels.sh   # cache ESM3 SS8 predictions
bash scripts/train_baseline.sh            # 120 epochs, paper_dispef_m.yaml
bash scripts/eval_baseline.sh             # evaluate on test set
```